# SKADI thumbnails

This notebook generates the thumbnails used in the SKADI user guide.

In [ ]:
import numpy as np
import plopp as pp
import scipp as sc

from ess.skadi import SkadiMcStasWorkflow
from ess.skadi.data import skadi_mcstas_sample
from ess.sans.types import Filename, RawDetector, SampleRun

In [ ]:
workflow = SkadiMcStasWorkflow()
workflow[Filename[SampleRun]] = skadi_mcstas_sample()
detector = workflow.compute(RawDetector[SampleRun])
detector_intensity = detector.bins.sum()

position = detector.coords["position"]


def normalize(coord):
    lower, upper = sc.min(coord), sc.max(coord)
    return (coord - 0.5 * (lower + upper)) / (upper - lower)


x, y, z = (normalize(position.fields[axis]) for axis in ("x", "y", "z"))
detector_intensity.coords["view_x"] = 0.82 * z + 0.57 * x
detector_intensity.coords["view_y"] = -0.15 * z + 0.21 * x + 0.97 * y

In [ ]:
def detector_view_plot():
    intensity = detector_intensity.values
    positive = intensity[intensity > 0]
    figure = pp.scatter(
        detector_intensity,
        x="view_x",
        y="view_y",
        cbar=True,
        logc=True,
        cmin=np.percentile(positive, 5),
        cmax=np.percentile(positive, 99.5),
        size=0.4,
        figsize=(3, 2.5),
        aspect="equal",
        rasterized=True,
        linewidths=0,
    )
    figure.ax.set_axis_off()
    figure.cax.set_axis_off()
    return figure

In [ ]:
fig = detector_view_plot()
fig.save(
    "../../docs/_static/thumbnails/skadi_detector_view_light.svg",
    transparent=True,
    bbox_inches="tight",
    pad_inches=0,
)
fig

In [ ]:
fig.save(
    "../../docs/_static/thumbnails/skadi_detector_view_dark.svg",
    transparent=True,
    bbox_inches="tight",
    pad_inches=0,
)
fig